# Setup and access check for connected examples

Use this short notebook when you want to diagnose the environment before opening the finance tutorial. It makes no LLM calls and publishes no prompts.

**Why this matters:** Python packages, Azure identity, Databricks workspace membership, and endpoint access are different prerequisites. Testing them in two checkpoints turns a later stack trace into an actionable setup message.

**What risk it prevents:** spending time debugging prompt or MLflow code when the real problem is the selected kernel, login, configuration, or endpoint.

**What evidence we will collect:** first, local package and storage readiness; second, cloud identity and endpoint readiness.

## Before you run it

From the repository root:

```bash
make examples-install
az login
cp aai-platform.example.yml aai-platform.yml  # only when absent
```

Configure `providers.models.general-chat.deployment`, then choose `<repository>/.venv/bin/python` in VS Code with **Select Kernel → Python Environments**.

The implementation lives in `examples/notebook_setup.py`. Both this notebook and `first_llm_call.ipynb` call that module directly. We intentionally do not use `%run`: the returned objects make dependencies visible, work with VS Code **Run All**, and do not rely on hidden state from another notebook.

In [ ]:
import importlib
import sys
from pathlib import Path

# VS Code may start the kernel at the repository root or inside examples/.
repo_root = next(
    (
        directory
        for directory in (Path.cwd(), *Path.cwd().parents)
        if (directory / "examples" / "notebook_setup.py").is_file()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError(
        "Open the cloned repository as your VS Code workspace, then restart "
        "the notebook kernel."
    )
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

setup_helpers = importlib.import_module("examples.notebook_setup")
environment = setup_helpers.prepare_notebook_environment(repo_root)

### Interpret `SETUP PASSED`

The selected kernel has the required packages and configuration. MLflow experiment, run, trace, and prompt metadata will use local SQLite; artifacts will use `.aai/local/mlruns`. Databricks will be used only for the model call unless a later tutorial section explicitly opts into a remote store.

## Verify cloud access without making an LLM call

**Why this matters:** authentication identifies you, while authorization determines which workspace and serving endpoint you may use.

This checkpoint checks the expected Azure tenant and subscription, workspace membership, the configured `general-chat` provider, and endpoint readiness. It does not register prompts, create experiment runs, or make a billable request.

In [ ]:
# Stop here with an actionable message before any billable model request.
connected = setup_helpers.preflight_databricks(environment)

### Interpret `PREFLIGHT PASSED`

Your Azure identity can enter the expected workspace and see a ready chat endpoint matching the configuration. Actual `CAN_QUERY` permission is proven only by a real request.

Next, open [`first_llm_call.ipynb`](first_llm_call.ipynb) and use **Run All**. It repeats these idempotent checks itself, so running this setup notebook first is helpful for diagnosis but never a hidden prerequisite.